---
# Byte Pair Encoding Tokenizer 
---

Let's build a BPE tokenizer for the coding dataset.

---
# Imports
---

Run the code cell below to import all relevant Python modules.

In [ ]:
import importlib
from torch.utils.data import DataLoader
import pandas as pd
import dataset
import instruction_dataset
import tests
import tokenizer
import visual
%matplotlib inline

---
## Byte Pair Encoding
---

A language model does not receive text strings directly. First, the text is split into **tokens**, and each token is represented by an integer id.

A byte-level BPE tokenizer starts with a very small vocabulary:

- one token for every possible byte value from `0` to `255`,
- a few special tokens with their own ids.

The special tokens have fixed jobs:

| Token | Meaning | Where it is used |
|---|---|---|
| `<pad>` | Padding token | Used  to make all sequences in a batch equally long. |
| `<bos>` | Beginning of sequence | Added at the start of a prompt-answer sequence, so the model has a clear first token. |
| `<eos>` | End of sequence | Added after the output, so LLM generation can learn when to stop. |
| `<unk>` | Unknown token | Kept for a standard tokenizer interface. A byte-level tokenizer almost never needs it, because every UTF-8 byte can be represented directly. |

This means the tokenizer can represent every UTF-8 string, including code, indentation, punctuation, emojis, and rare symbols. It never has to give up because it does not know a word.

The problem is that byte sequences are long. For example, the text `return` initially starts as six separate byte tokens. BPE shortens common patterns by learning **merge rules**.

BPE training repeats the same three steps:

1. Count all neighboring token pairs.
2. Pick the most frequent pair.
3. Replace every occurrence of that pair with a new token id.

A tiny example:

```text
texts:       low low lower
start:       l o w   l o w   l o w e r
merge 1:     lo w    lo w    lo w e r      # merge (l, o)
merge 2:     low     low     low e r       # merge (lo, w)
```

After training, 
- `encoding` starts from UTF-8 bytes again and applies the learned merges in the same order. 
- `decoding` does the reverse: each learned token is **recursively** expanded back into the bytes it represents, then the bytes are decoded as UTF-8.

---

## Architecture

---

In this exercise, the exact internal helper functions are up to you. The rest of the code only relies on the tokenizer interface.

Your `BytePairTokenizer` class should provide this interface:

```python
tokenizer = BytePairTokenizer()

tokenizer.pad_id      # int, id of <pad>
tokenizer.bos_id      # int, id of <bos>
tokenizer.eos_id      # int, id of <eos>
tokenizer.unk_id      # int, id of <unk>
tokenizer.vocab_size  # int, current vocabulary size
tokenizer.merges      # list, pairs of token ids


tokenizer.train(texts, vocab_size=1024, min_frequency=2)
tokenizer.encode(text, add_bos=False, add_eos=False)
tokenizer.decode(token_ids, skip_special_tokens=True)
tokenizer.token_to_bytes(token_id)
```

Expected behavior:

- `train` learns BPE merges from an **iterable of strings** (`for text in texts:`) and returns the tokenizer object itself.
- `encode` returns a Python `list` of integer token ids.
- `decode` returns a Python string and should reconstruct the original text after `encode`, ignoring special tokens by default.
- `token_to_bytes` returns the byte values (as `list`) represented by one token id. This is needed for decoding and visualization.
- The tokenizer should store learned merges in `tokenizer.merges` as list of tuples `(int, int)` as in the example above, so we can inspect them later.
- The special tokens need their own token ids and are **not** converted to UTF-8 bytes.

## **Task:**

Implement `BytePairTokenizer` in `tokenizer.py`.

**Hints:**

- The class should provide the attributes `pad_id`, `bos_id`, `eos_id`, `unk_id`, `merges` and `vocab_size`.
- The class should provide the methods `train`, `encode`, `decode`, and `token_to_bytes`.
- You may choose your own internal helper methods for counting and replacing pairs.
- Use `string.encode("utf-8")` to get the UTF-8 encoding of the text.
- Start from byte tokens, not words. This makes the tokenizer work for arbitrary code and text.
- In `train`, repeatedly count pairs, select the most common pair, and replace it with a new token id. 
- The vocabulary gets larger during `train`.

Run the cell below.

In [ ]:
importlib.reload(tokenizer)
importlib.reload(dataset)
importlib.reload(tests)

tests.test_byte_pair_tokenizer_interface()
tests.test_special_tokens()
tests.test_bpe_learns_expected_merge()

---
## Load the Code Instruction Dataset
---

In [ ]:
data = instruction_dataset.InstructionDataset(r"D:\Git\13-Large-Language-Models\13b-tokenizer\code_instructions_120k.arrow")# print("First row of dataset:", data[0])
print("Length of dataset:", len(data))
print("Column names:", data.column_names)
print("First row of dataset:", data[0])

---
## Training Data Formatting
---

Before tokenization, each sample from the code instruction dataset is converted to one string for training:

```text
### Instruction:
[here instruction content]

### Input:
[here input content]

### Output:
[here output content]
```

If the `input` field is empty, the input section is skipped.

## **Task:**

Implement `format_instruction_prompt` and `format_instruction_text` in `dataset.py`.

**Hints:**

- Strip whitespace from the instruction and input fields.
- Only include the `### Input:` block if the input field is not empty.
- The prompt should end with `### Output:\n`.
- `format_instruction_text` should append the stripped output and one final newline.

Run the cell below.

In [ ]:
importlib.reload(dataset)
importlib.reload(tests)
tests.test_prompt_formatting()

Show the formatting:

In [ ]:
bpe = dataset.train_bpe_tokenizer(
    data,
    vocab_size=512,
    min_frequency=2,
    max_examples=500,
)

bpe.vocab_size, len(bpe.merges)

tests.test_bpe_learns_expected_merge()
tests.test_encode_decode_roundtrip()


---
## Encode and Decode
---

- Encoding maps text to token ids. 
- Decoding maps token ids back to text.

You already implemented this behavior as part of `BytePairTokenizer`. Now we run a small sanity check:

- `add_bos=True` should prepend the beginning-of-sequence token.
- `add_eos=True` should append the end-of-sequence token.
- Decoding should skip special tokens by default.

Run the cell below.


In [ ]:
text = "def add(a, b):\n    return a + b"
token_ids = bpe.encode(text, add_bos=True, add_eos=True)

print("Original:\n")
print("--------------------------------")
print(text)
print("--------------------------------")
print("\nToken IDs:\n")
print("--------------------------------")
print(token_ids[:30])
print("--------------------------------")
print("\nDecoded Tokens:\n")
print("--------------------------------")
print(bpe.decode(token_ids))
print("--------------------------------")

tests.test_encode_decode_roundtrip()


---
## Visualize Tokenization
---

Each colored block is one token. Hover over a token to see its token id and byte length. 

If the tokens look strange, inspect your merge order, `encode`, or `token_to_bytes`.

Run the cells below.


In [ ]:
text = "def bubblesort(arr):\n    n = len(arr)\n    for i in range(n):\n        for j in range(0, n-i-1):\n            if arr[j] > arr[j+1]:\n                arr[j], arr[j+1] = arr[j+1], arr[j]\n    return arr"
print("Original:\n")
print("--------------------------------")
print(text)
print("--------------------------------")
print("\n Tokens (colours have no meaning, but it looks cool):")
visual.show_tokenization(bpe, text)

In [ ]:
pd.DataFrame(visual.token_table(bpe, text)).head(12)

---
## Tokenizer Statistics
---

The next plots show how long examples become after tokenization and how large the learned merge tokens are.

- Token counts tell us how long training sequences become.
- Merge byte lengths show whether BPE learned longer recurring fragments.

Run the cells below.


In [ ]:
visual.plot_token_count_histogram(
    bpe,
    dataset.iter_instruction_texts(data),
    max_examples=300,
);

In [ ]:
visual.plot_merge_byte_lengths(bpe, max_merges=512);

---
## Tokenized Dataset
---

The raw code dataset stores three strings: 
- an instruction, 
- optional input, 
- and the desired output.

But, the model trains on tokens, so a token-based dataset is needed.

This dataset turns one row into two fixed-size tensors:

- `model_input`: the token ids fed into the Transformer.
- `labels`: the next token ids the model should predict.

For next-token prediction, we first build one full token sequence:

`<bos>` + prompt + output + `<eos>`

Then we shift it by one position. The model input sees every token except the last one, and the labels contain every token except the first one.

In addition, we only want to train on the answer, not on the prompt text that was given to the model. 

PyTorch's cross-entropy loss ignores label values equal to `-100`, so prompt labels and padding labels are set to `-100`.

Finally, every example is padded or truncated to a fixed length `max_length`. Padding is needed because a batch is one tensor, and tensors need a rectangular shape: every row in the batch must have the same length. Short examples get `<pad>` tokens in `model_input`, while their matching `labels` stay at `-100` so the padding never contributes to the loss.

## **Task:**

Implement `BPE_Token_Dataset.__getitem__` in `dataset.py`.

**Hints:**

- Encode the prompt with a BOS token.
- Encode the output with an EOS token.
- Keep at most `max_length + 1` tokens before shifting.
- `model_input` should contain all tokens except the last token.
- `labels` should contain all tokens except the first token.
- Pad `model_input` to `max_length` with `pad_id`.
- Pad `labels` to `max_length` with `-100`.
- Set prompt labels to `-100` so they do not contribute to the loss.
- Useful function: `torch.full` with `dtype=torch.long`.
- Look up PyTorch's cross-entropy loss.

Run the cell below.

In [ ]:
importlib.reload(dataset)
importlib.reload(tests)
tests.test_bpe_dataset()
token_data = dataset.BPE_Token_Dataset(data, bpe, max_length=384)
example = token_data[0]
{name: tuple(value.shape) for name, value in example.items() if hasattr(value, "shape")}

---
## Visual Check - Debugging
---

The next views make dataset bugs visible.

- Red tokens are prompt targets ignored by the loss.
- Green tokens are output targets used by the loss.
- Gray positions are padding ignored by the loss.

In a healthy example, green starts after `### Output:` and padding stays gray.

In [ ]:
importlib.reload(visual)
visual.show_tokenized_example(bpe, token_data[3])

Have a look why padding is necessary for optimization with batches. Let's build a batch:

In [ ]:
loader = DataLoader(token_data, batch_size=12, shuffle=True)
batch = next(iter(loader))
visual.show_tokenized_batch(bpe, batch, max_rows=12, max_cols=200)